In [1]:
# Imports / global contants

# csv Dateien sind im Verzeichnis ../data zu finden

import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import math
import numpy as np
import seaborn as sns

from pandas.plotting import parallel_coordinates

timeFormat = "%Y-%m-%dT%H:%M:%S.%fZ"

export = "../export/"
export_img = "../export/img/"

path = "../data/"

percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]

In [2]:
df_study = pd.read_csv(rf'{export}data_complete.csv', sep= ";")
display(df_study)

,Unnamed: 0,DateTime,State,mappingMethod,TaskNo,TargetLayers,LayerCount,TrialIdx,posX,posY,posZ,TimeStamp,iteractionType,currentLayer,Proband,FrameTime
0,0,2021-06-29T07:13:59.622Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,NaN
1,1,2021-06-29T07:13:59.630Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,8.0
2,2,2021-06-29T07:13:59.673Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,43.0
3,3,2021-06-29T07:13:59.707Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,34.0
4,4,2021-06-29T07:13:59.742Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2733075,2733075,2021-05-31T10:19:26.341Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12,32.0
2733076,2733076,2021-05-31T10:19:26.416Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12,75.0
2733077,2733077,2021-05-31T10:19:26.444Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12,28.0
2733078,2733078,2021-05-31T10:19:26.474Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12,30.0


In [3]:
df_cleaned = pd.read_csv(rf'{export}cleanedStudy_interactionOnly_removed-inactive-300.csv', sep= ";")
display(df_cleaned)

,DateTime,Unnamed: 0,SampleIdx_global,SampleIdx_Proband,State,mappingMethod,Task,TargetLayers,NumLayers,Trial,...,iteractionType,currentLayer,Proband,FrameTime,shifted,Result,Block,Target,Target_Relative,DateTime2
0,2021-06-29 07:19:27.465000+00:00,0,65,65,VIEW,direct,1,"['3', '11', '9', '6', '1']",12,0,...,NaN,NaN,23,4.0,1.0,START,1,3,0.250000,2021-06-29 07:19:27.465000+00:00
1,2021-06-29 07:19:27.498000+00:00,1,66,66,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,1,23,33.0,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.498000+00:00
2,2021-06-29 07:19:27.540000+00:00,2,67,67,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,1,23,42.0,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.540000+00:00
3,2021-06-29 07:19:27.598000+00:00,3,68,68,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,-,23,58.0,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.598000+00:00
4,2021-06-29 07:19:27.637000+00:00,4,69,69,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,-,23,39.0,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.637000+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1398926,2021-05-31 10:18:49.509000+00:00,1535230,2360990,90024,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,32.0,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.509000+00:00
1398927,2021-05-31 10:18:49.544000+00:00,1535231,2360991,90025,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,35.0,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.544000+00:00
1398928,2021-05-31 10:18:49.571000+00:00,1535232,2360992,90026,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,27.0,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.571000+00:00
1398929,2021-05-31 10:18:49.602000+00:00,1535233,2360993,90027,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,31.0,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.602000+00:00


In [4]:
def computeCI(df, m= 'mean', c='count', s='std'):
    df['ci95_hi'] = df[m] + 1.96*df[s]/(df[c].apply(np.sqrt))
    df['ci95_lo'] = df[m] - 1.96*df[s]/(df[c].apply(np.sqrt))

def computeWhiskers(df, desc, col_names, q1='25%', q3='75%'):
    desc['iqr'] = desc[q3] - desc[q1]

    # Whisker-Berechnung (innerhalb 1.5 * IQR)
    desc['lower_bound'] = desc[q1] - 1.5 * desc['iqr']
    desc['upper_bound'] = desc[q3] + 1.5 * desc['iqr']    

    lower_whiskers = []
    upper_whiskers = []

    for col_name in col_names:
        lower_bound = desc['lower_bound'].T[col_name]
        upper_bound = desc['upper_bound'].T[col_name]

        # Whiskers sind die letzten Punkte innerhalb dieser Grenzen
        l = df[df[col_name] >= lower_bound][col_name].min()
        u = df[df[col_name] <= upper_bound][col_name].max()

        lower_whiskers.append(l)
        upper_whiskers.append(u)

    desc['lower_whisker'] = lower_whiskers
    desc['upper_whisker'] = upper_whiskers

In [5]:
df_study['FrameTime'] = pd.to_datetime(df_study["DateTime"], format=timeFormat).diff().dt.microseconds / 1000.0

In [6]:
desc_complete = pd.DataFrame(df_study['FrameTime'].describe(percentiles=percentiles)).T

computeCI(desc_complete)

computeWhiskers(df_study, desc_complete, ['FrameTime'])

display(desc_complete.T)

desc_complete.T.to_csv(rf'{export}desc-stats_framestats_complete.csv', sep=';')


,FrameTime
count,2.733079e+06
mean,3.750637e+01
std,8.842094e+00
min,0.000000e+00
5%,2.700000e+01
10%,2.900000e+01
25%,3.200000e+01
50%,3.600000e+01
75%,4.300000e+01
90%,4.900000e+01


In [7]:
desc_interaction = pd.DataFrame(df_cleaned['FrameTime'].describe(percentiles=percentiles)).T

computeCI(desc_interaction)

computeWhiskers(df_cleaned, desc_interaction, ['FrameTime'])

display(desc_interaction.T)

desc_interaction.T.to_csv(rf'{export}desc-stats_framestats_interaction.csv', sep=';')

,FrameTime
count,1.398931e+06
mean,3.742710e+01
std,8.674817e+00
min,0.000000e+00
5%,2.600000e+01
10%,2.900000e+01
25%,3.200000e+01
50%,3.600000e+01
75%,4.300000e+01
90%,4.800000e+01


In [8]:
df_framerates = pd.read_csv(rf'{export}times.csv', sep= ";")
display(df_framerates)

,Unnamed: 0,MappingMethod,FrameNumber_Start,FrameNumber_Finish,Block,Task,Trial,NumLayers,Target,Target_Relative,DateTime_Start,DateTime_Finish,Duration,NumFrames,Result,Proband,FrameRate_1
0,0,direct,65,242,1,1,0,12,3,0.250000,2021-06-29T07:19:27.465Z,2021-06-29T07:19:34.185Z,6.720,177,COMPLETED,23,26.339286
1,1,direct,295,547,1,1,1,12,11,0.916667,2021-06-29T07:19:36.096Z,2021-06-29T07:19:45.693Z,9.597,252,COMPLETED,23,26.258206
2,2,direct,604,808,1,1,2,12,9,0.750000,2021-06-29T07:19:47.749Z,2021-06-29T07:19:55.418Z,7.669,204,COMPLETED,23,26.600600
3,3,direct,849,1050,1,1,3,12,6,0.500000,2021-06-29T07:19:56.961Z,2021-06-29T07:20:04.496Z,7.535,201,TERMINATED,23,26.675514
4,4,direct,1114,1350,1,1,4,12,1,0.083333,2021-06-29T07:20:06.928Z,2021-06-29T07:20:16.018Z,9.090,236,COMPLETED,23,25.962596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6205,6205,widening,88872,89061,3,18,0,15,8,0.533333,2021-05-31T10:18:07.589Z,2021-05-31T10:18:14.593Z,7.004,189,COMPLETED,12,26.984580
6206,6206,widening,89096,89278,3,18,1,15,11,0.733333,2021-05-31T10:18:15.806Z,2021-05-31T10:18:22.355Z,6.549,182,COMPLETED,12,27.790502
6207,6207,widening,89320,89518,3,18,2,15,14,0.933333,2021-05-31T10:18:23.773Z,2021-05-31T10:18:31.158Z,7.385,198,TERMINATED,12,26.811104
6208,6208,widening,89570,89735,3,18,3,15,4,0.266667,2021-05-31T10:18:32.948Z,2021-05-31T10:18:38.840Z,5.892,165,TERMINATED,12,28.004073


In [9]:
desc_framerate = pd.DataFrame(df_framerates['FrameRate_1'].describe(percentiles=percentiles)).T

computeCI(desc_framerate)

computeWhiskers(df_framerates, desc_framerate, ['FrameRate_1'])

display(desc_framerate.T)

desc_framerate.T.to_csv(rf'{export}desc-stats_framerate.csv', sep=';')

,FrameRate_1
count,6210.000000
mean,26.609964
std,0.623125
min,17.328951
5%,25.859548
10%,26.027958
25%,26.285908
50%,26.589875
75%,26.914084
90%,27.208423


In [10]:
df_framerates['FrameTime'] = 1.0 / df_framerates['FrameRate_1'] * 1000

desc_frametime = pd.DataFrame(df_framerates['FrameTime'].describe(percentiles=percentiles)).T

computeCI(desc_frametime)

computeWhiskers(df_framerates, desc_frametime, ['FrameTime'])

display(desc_frametime.T)

desc_frametime.T.to_csv(rf'{export}desc-stats_frametime.csv', sep=';')

,FrameTime
count,6210.000000
mean,37.597718
std,0.793343
min,19.123016
5%,36.492932
10%,36.753325
25%,37.155268
50%,37.608300
75%,38.043198
90%,38.420225
